# 🚀 TaleForge: 1-Click Free GPU LoRA Training
### Train your own private AI model on your Bengali stories using Google Colab's Free T4 GPU (15 GB VRAM).

**Instructions:**
1. In the menu, click **Runtime > Change runtime type > T4 GPU**.
2. Run the cells step-by-step (Shift + Enter).
3. Upload your `train.jsonl` exported from TaleForge (`data/datasets/train.jsonl`).
4. Download the trained LoRA adapter weights (`taleforge-lora.zip`) and place them in `models/adapters/` in your TaleForge repository!

In [ ]:
# Step 1: Verify GPU
!nvidia-smi

In [ ]:
# Step 2: Install ML Libraries
!pip install -q -U torch transformers peft datasets accelerate bitsandbytes trl

In [ ]:
# Step 3: Upload your TaleForge train.jsonl dataset
from google.colab import files
import os

print("Upload your 'train.jsonl' from TaleForge/data/datasets/:")
uploaded = files.upload()
dataset_filename = list(uploaded.keys())[0]
print(f"Loaded dataset: {dataset_filename}")

In [ ]:
# Step 4: Configure Base Model & Quantization
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

# 1.5B runs on a 4 GB home GPU. 3B follows instructions noticeably better (still trains on the free T4)
# but needs ~3 GB VRAM in 4-bit at home. 7B needs a cloud GPU for inference.
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # or "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "taleforge-lora"

print(f"Loading {BASE_MODEL} in 4-bit precision...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# Apply LoRA (Low-Rank Adaptation)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Step 5: Format Dataset and Train
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

raw_dataset = load_dataset("json", data_files={"train": dataset_filename})["train"]

def tokenize_format(example):
    messages = example.get("messages")
    if messages and hasattr(tokenizer, "apply_chat_template"):
        prompt_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
        full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    else:
        prompt_text = f"User: {example.get('instruction', '')}\nAssistant: "
        full_text = prompt_text + example.get("output", "") + (tokenizer.eos_token or "")

    # No padding here: the collator pads each batch dynamically and fills label padding with -100
    # 1792 tokens: multi-turn samples (short draft + 'make it longer' + full chunk) need more than 1024 (longest ~1550)
    tokenized = tokenizer(full_text, truncation=True, max_length=1792)
    prompt_len = len(tokenizer(prompt_text, truncation=True, max_length=1792)["input_ids"])

    # Only the story (assistant reply) is learned; system/user prompt tokens are ignored in the loss
    labels = list(tokenized["input_ids"])
    labels[:prompt_len] = [-100] * prompt_len
    tokenized["labels"] = labels
    return tokenized

tokenized_ds = raw_dataset.map(tokenize_format, remove_columns=raw_dataset.column_names)
# Drop samples whose prompt filled the whole window (no story tokens left to learn from)
tokenized_ds = tokenized_ds.filter(lambda ex: any(label != -100 for label in ex["labels"]))
print(f"Training samples: {len(tokenized_ds)}")

training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=1,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True),
)

print("Starting LoRA training...")
trainer.train()

# Save LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter successfully saved to {OUTPUT_DIR}!")

In [ ]:
# Step 6: Test Generation with your newly trained AI Model!
prompt = "একটি বৃষ্টির রাতের রোমান্টিক গল্প রচনা করো"
# Same system prompt the dataset was built with (apps/web/lib/system-prompt.ts)
SYSTEM_PROMPT = "You are TaleForge AI, an expert literary novelist specializing in Bengali literature. Follow the user's latest instruction exactly. If they ask to expand, continue, shorten, rewrite or change the previous story, work on that story from the conversation and keep its title, characters and events. Otherwise write a new story. Write in Bengali unless the user asks for English."
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": prompt}
]
text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text_input], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.1,
)

generated_story = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Generated Story from Your Trained Model ---\n")
print(generated_story)

# Follow-up test: the model must EXPAND the story above, not write a new one
messages += [
    {"role": "assistant", "content": generated_story},
    {"role": "user", "content": "ei golpo ta aro boro koro"},
]
text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text_input], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=768, temperature=0.8, top_p=0.9, repetition_penalty=1.1)
print("\n--- Expanded Story (follow-up instruction) ---\n")
print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

In [ ]:
# Step 7: Download your trained LoRA adapter to your computer
import shutil
shutil.make_archive("taleforge-lora", 'zip', OUTPUT_DIR)
files.download("taleforge-lora.zip")
print("Extract the zip contents into TaleForge/models/adapters/taleforge-lora in your project!")

## Step 8 (free hosting): merge, compress and publish the model
<!-- TALEFORGE_EXPORT_CELL -->
These cells turn the trained adapter into a single compressed model file (GGUF) that runs on a **free Hugging Face Space CPU**, and upload it to your Hugging Face account.

You need a free Hugging Face account and a **write** token from https://huggingface.co/settings/tokens. When asked, paste the token. Then follow `ai/deploy/README.md` (section *Free route*) to create the Space.

In [ ]:
# Step 8a: merge the LoRA adapter into the base model (full weights, fp16)
# TALEFORGE_EXPORT_CELL
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MERGED_DIR = "taleforge-merged"

# Free the 4-bit training model, then reload the base in fp16 for a clean merge
try:
    del model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="cpu", trust_remote_code=True)
merged = PeftModel.from_pretrained(base, OUTPUT_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True).save_pretrained(MERGED_DIR)
print("Merged model saved to", MERGED_DIR)

In [ ]:
# Step 8b: convert to GGUF and quantize to Q4_K_M (small and fast enough for a free CPU Space)
# TALEFORGE_EXPORT_CELL
import os, subprocess

if not os.path.isdir("llama.cpp"):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

F16_FILE = "taleforge-lora.f16.gguf"
Q4_FILE = "taleforge-lora.Q4_K_M.gguf"
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {F16_FILE} --outtype f16

# Build only the quantizer (2-4 minutes). If the build fails, the f16 file is used instead (bigger, slower).
GGUF_FILE = F16_FILE
try:
    subprocess.run(["cmake", "-S", "llama.cpp", "-B", "llama.cpp/build", "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF"], check=True, capture_output=True)
    subprocess.run(["cmake", "--build", "llama.cpp/build", "--target", "llama-quantize", "-j"], check=True, capture_output=True)
    subprocess.run(["llama.cpp/build/bin/llama-quantize", F16_FILE, Q4_FILE, "Q4_K_M"], check=True)
    GGUF_FILE = Q4_FILE
except Exception as exc:
    print("Quantization skipped:", exc)
print("Model file:", GGUF_FILE, f"({os.path.getsize(GGUF_FILE) / 1e9:.2f} GB)")

In [ ]:
# Step 8c: upload the model file to your Hugging Face account
# TALEFORGE_EXPORT_CELL
from huggingface_hub import HfApi, login, whoami

login()  # paste a WRITE token from https://huggingface.co/settings/tokens
username = whoami()["name"]
MODEL_REPO = f"{username}/taleforge-lora-gguf"

api = HfApi()
api.create_repo(MODEL_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_file(path_or_fileobj=GGUF_FILE, path_in_repo=GGUF_FILE, repo_id=MODEL_REPO, repo_type="model")
print("Uploaded. Set these on your Space:")
print("  MODEL_REPO =", MODEL_REPO)
print("  MODEL_FILE =", GGUF_FILE)